# 🌸 Nayari AI — Kaggle Export Notebook
> Use this notebook to export your trained Nayari AI model to different formats.

## 📦 Step 1 — Install

In [1]:
%%capture
import os

# 1. Clear out the conflicting versions completely
os.system('pip uninstall -y torch torchvision torchaudio xformers bigframes s3fs gcsfs')

# 2. Install the Torch 2.10.0 stack (Compatible with Unsloth 2026.5.2)
# We use +cu128 or +cu124 depending on Kaggle's current backend. 
# +cu128 is generally the standard for May 2026 kernels.
os.system(
    'pip install -q --upgrade '
    'torch==2.10.0 '
    'torchvision==0.25.0 '
    'torchaudio==2.10.0 '
    'xformers '
    '--index-url https://download.pytorch.org/whl/cu128'
)

# 3. Install Unsloth and matching ecosystem tools
# Note: Using unsloth[kaggle-new] helps with specific Kaggle optimizations
os.system('pip install -q "unsloth[kaggle-new]"')

# 4. Upgrade the rest of the stack to the versions Unsloth expects
os.system(
    'pip install -q --upgrade '
    '"trl>=0.18.2,<=0.24.0" '
    '"transformers>=4.51.3,<=5.5.0" '
    '"datasets>=3.4.1,<4.4.0" '
    'accelerate peft bitsandbytes '
    '"fsspec>=2025.2.0"'
)

# 5. Verify the installation
import torch
print(f"Torch version: {torch.__version__}")
try:
    import unsloth
    print("Unsloth imported successfully!")
except Exception as e:
    print(f"Import failed: {e}")


Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: xformers 0.0.35
Uninstalling xformers-0.0.35:
  Successfully uninstalled xformers-0.0.35


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 14.0 MB/s eta 0:00:00


## 🔄 Step 1.5 — Reload Model for Export *(run only if skipping training)*
> Skip this cell if you just finished Step 7 (training) — `model` is already in memory.
>
> Run this cell **only** if you restarted the session or want to re-export without retraining.
> It auto-selects the best available checkpoint in order:
> 0. **Custom path override** — set `CUSTOM_PATH` to load from any folder directly
> 1. Merged 16-bit weights (`nayari_merged_16bit`) — fastest load
> 2. LoRA adapters (`nayari_lora`)
> 3. Latest training checkpoint (`nayari_checkpoints/checkpoint-*` or `nayari_stabilize/checkpoint-*`)

In [6]:
# ── Step 1.5: Reload model + tokenizer for export (skip if already in memory)
# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048  # must match Step 4

# ╔══════════════════════════════════════════════════════════════════╗
# ║  LOAD PATHS — edit these to point at your saved artifacts       ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  CUSTOM_PATH  → set to any folder to bypass auto-detection      ║
# ║                 (Priority 0 — overrides everything below)       ║
# ║  Leave as "" to use the automatic priority cascade below.       ║
# ╚══════════════════════════════════════════════════════════════════╝
CUSTOM_PATH = ""  # e.g. "/kaggle/working/nayari_checkpoints/checkpoint-500"

MERGED_PATH       = "/kaggle/working/nayari_merged_16bit"
LORA_PATH         = "/kaggle/working/nayari_lora"
CHECKPOINT_ROOTS  = [                           # searched in order; first hit wins
    "/kaggle/working/nayari_checkpoints",
    "/kaggle/working/nayari_stabilize",
]

# ── helpers ───────────────────────────────────────────────────────────────────
def _already_loaded():
    """Return True if model/tokenizer are already in memory from training."""
    try:
        _ = model  # noqa: F821
        _ = tokenizer  # noqa: F821
        return True
    except NameError:
        return False

def _latest_checkpoint(root):
    """Return the path of the highest-numbered checkpoint-N folder, or None."""
    p = Path(root)
    if not p.exists():
        return None
    ckpts = sorted(
        [d for d in p.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    return str(ckpts[-1]) if ckpts else None

def _best_checkpoint(roots):
    """Search each root in order; return the first checkpoint found, or None."""
    for root in roots:
        ckpt = _latest_checkpoint(root)
        if ckpt:
            return ckpt
    return None

# ── main logic ────────────────────────────────────────────────────────────────
if _already_loaded():
    print("Model already in memory — skipping reload.")
else:
    load_path = None
    label     = None

    # Priority 0: user-specified custom path (overrides everything)
    if CUSTOM_PATH.strip():
        if not Path(CUSTOM_PATH).exists():
            raise FileNotFoundError(f"CUSTOM_PATH not found: {CUSTOM_PATH}")
        load_path = CUSTOM_PATH.strip()
        label = f"custom path ({Path(load_path).name})"

    # Priority 1: merged 16-bit (best for GGUF / push to HF)
    elif Path(MERGED_PATH).exists():
        load_path = MERGED_PATH
        label = "merged 16-bit"

    # Priority 2: LoRA adapters
    elif Path(LORA_PATH).exists():
        load_path = LORA_PATH
        label = "LoRA adapters"

    # Priority 3: latest checkpoint across all configured roots
    else:
        ckpt = _best_checkpoint(CHECKPOINT_ROOTS)
        if ckpt:
            load_path = ckpt
            label = f"checkpoint ({Path(load_path).name} in {Path(load_path).parent.name})"

    if not load_path:
        roots_str = "\n".join(f"  {r}/checkpoint-*/" for r in CHECKPOINT_ROOTS)
        raise FileNotFoundError(
            "No saved model found! Expected one of:\n"
            f"  {MERGED_PATH}\n"
            f"  {LORA_PATH}\n"
            f"{roots_str}\n"
            "Or set CUSTOM_PATH to your checkpoint folder."
        )

    print(f"Loading from {label}: {load_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=load_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,  # must be False for correct GGUF export
    )
    print(f"✅ Loaded from {label}")


Model already in memory — skipping reload.


## 💾 Step 2 — Save & Export

> **⚠️ Ensure Step 4 uses `load_in_4bit=False`** (already set).  
> With 4-bit quantisation the base weights cannot be dequantized, so the exported GGUF / merged model
> will contain only the generic base model — your fine-tuning is silently lost.

Run **one or more** of the blocks below. Each is independent.

| Block | Output | Size | Best for |
|---|---|---|---|
| **2A** | LoRA adapters | ~50 MB | Further training, merging later |
| **2B** | Merged 16-bit | ~3 GB | Full inference, HuggingFace upload |
| **2C** | GGUF Q4_K_M | ~1 GB | Ollama / LM Studio / llama.cpp |
| **2D** | GGUF Q8_0 | ~1.7 GB | Higher quality local inference |
| **2E** | HTTP Download | — | Tunnel download from Kaggle |
| **2F** | HuggingFace Hub | — | Permanent hosting at `Crossie/Nayari` |


### 2A — LoRA Adapters Only *(smallest, ~50 MB)*

In [4]:
LORA_PATH = "/kaggle/working/nayari_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

import os
size_mb = sum(f.stat().st_size for f in Path(LORA_PATH).rglob("*") if f.is_file()) / 1024**2
print(f"✅ LoRA adapters saved → {LORA_PATH}")
print(f"   Size: {size_mb:.1f} MB")
print("   Load later with: model, tokenizer = FastLanguageModel.from_pretrained(LORA_PATH)")


NameError: name 'model' is not defined

### 2B — Merged 16-bit Model *(full weights, ~3 GB)*

In [4]:
MERGED_PATH = "/kaggle/working/nayari_merged_16bit"

# save_method="merged_16bit" merges LoRA adapters into base model weights and
# saves as a standard HuggingFace model in bfloat16 precision.
model.save_pretrained_merged(
    MERGED_PATH, tokenizer,
    save_method="merged_16bit",
)

size_gb = sum(f.stat().st_size for f in Path(MERGED_PATH).rglob("*") if f.is_file()) / 1024**3
print(f"✅ Merged 16-bit saved → {MERGED_PATH}")
print(f"   Size: {size_gb:.2f} GB  (includes full merged weights — this is the fine-tuned model)")
print("   Use with: AutoModelForCausalLM.from_pretrained(MERGED_PATH)")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/kaggle/working/nayari_merged_16bit`: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]


Successfully copied all 1 files from cache to `/kaggle/working/nayari_merged_16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:17<00:00, 17.22s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/nayari_merged_16bit`
✅ Merged 16-bit saved → /kaggle/working/nayari_merged_16bit
   Size: 2.89 GB  (includes full merged weights — this is the fine-tuned model)
   Use with: AutoModelForCausalLM.from_pretrained(MERGED_PATH)


### 2C — GGUF Q4_K_M *(quantised, ~1 GB — Ollama / LM Studio)*

In [7]:
GGUF_Q4_PATH = "/kaggle/working/nayari_gguf_q4"

model.save_pretrained_gguf(
    GGUF_Q4_PATH, tokenizer,
    quantization_method="q4_k_m",
)

# Rename to clean nayari-Q4_K_M.gguf (handles duplicates safely)
for f in sorted(Path(GGUF_Q4_PATH).rglob("*.gguf")):
    target = f.parent / "nayari-Q4_K_M.gguf"
    if f.name == target.name:
        print(f"Already named: {f} ({f.stat().st_size/1024**3:.2f} GB)")
        continue
    if target.exists():
        target.unlink()
    f.rename(target)
    print(f"GGUF Q4_K_M -> {target} ({target.stat().st_size/1024**3:.2f} GB)")
    break  # only rename the first one
print("   Load in Ollama: ollama create nayari -f Modelfile")
print("   Load in LM Studio: import the .gguf file directly")


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/kaggle/working/nayari_gguf_q4`: 100%|██████████| 1/1 [00:04<00:00,  4.08s/it]


Successfully copied all 1 files from cache to `/kaggle/working/nayari_gguf_q4`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:18<00:00, 18.21s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/nayari_gguf_q4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will build
Unsloth: Updating system package directories
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/kaggle/working/nayari_gguf_q4_gguf/Nayari.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This mig

### 2D — GGUF Q8_0 *(higher quality, ~1.7 GB)*

In [ ]:
# 9D reloads from the already-merged 16-bit weights saved in 9B.
# This avoids the transformers revert_weight_conversion bug that fires
# when save_pretrained_gguf is called on a live PEFT / for_inference model.
# Run 9B before this cell.
GGUF_Q8_PATH = "/kaggle/working/nayari_gguf_q8"
MERGED_PATH  = "/kaggle/working/nayari_merged_16bit"

from unsloth import FastLanguageModel as _FLM
model_q8, tok_q8 = _FLM.from_pretrained(
    model_name=MERGED_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)
model_q8.save_pretrained_gguf(
    GGUF_Q8_PATH, tok_q8,
    quantization_method="q8_0",
)

# Rename to clean nayari-Q8_0.gguf
for f in sorted(Path(GGUF_Q8_PATH).rglob("*.gguf")):
    new_name = f.parent / "nayari-Q8_0.gguf"
    f.rename(new_name)
    print(f"✅ GGUF Q8_0 → {new_name}  ({new_name.stat().st_size/1024**3:.2f} GB)")


==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: Model is not a PEFT model. Using existing checkpoint at /kaggle/working/nayari_merged_16bit
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


### 2F — Push Everything to HuggingFace (`Crossie/Nayari`)
Pushes **merged 16-bit weights + GGUF Q4_K_M + GGUF Q8_0** in one go.

> **Before running:** Add your HuggingFace Write token to Kaggle Secrets:
> Kaggle → **Add-ons → Secrets** → **Add Secret** → Name: `HF_TOKEN`
> ([Get token](https://huggingface.co/settings/tokens) — needs **Write** scope)

> Run **Step 9B** (merged 16-bit save) before this cell.

In [4]:
import os
from pathlib import Path
from huggingface_hub import login, HfApi

REPO_ID    = "Crossie/Nayari"
IS_PRIVATE = False

# =====================================================================
#  UPLOAD SETTINGS
# =====================================================================
UPLOAD_MODE      = "weights_only" # "weights_only" or "everything"
REPLACE_EXISTING = False         # If True, overwrites. If False, skips if file exists.

# -- Auth -----------------------------------------------------------------
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "").strip()

if not hf_token:
    raise RuntimeError("HF_TOKEN not found!")

login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
api.create_repo(repo_id=REPO_ID, private=IS_PRIVATE, exist_ok=True)

# Get current list of files in repo to decide whether to skip
repo_files = api.list_repo_files(repo_id=REPO_ID)

def safe_rename_gguf(directory, target_name):
    gguf_files = sorted(Path(directory).rglob("*.gguf"))
    if not gguf_files: return []
    src = gguf_files[0]
    dst = src.parent / target_name
    if src != dst:
        if dst.exists(): dst.unlink()
        src.rename(dst)
    return [dst]

# -- 1. Upload merged 16-bit weights --------------------------------------
MERGED_PATH = "/kaggle/working/nayari_merged_16bit"
if Path(MERGED_PATH).exists():
    print(f"\nChecking merged 16-bit model folder...")
    # upload_folder is smart: it only uploads changed/new files (LFS hashing)
    # If REPLACE_EXISTING is True, we don't need to do anything special, it just commits.
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=REPO_ID,
        commit_message="Update merged 16-bit model",
        # delete_patterns="*" would clear repo first; usually better to just sync:
    )
    print("Merged weights synced.")

if UPLOAD_MODE == "everything":
    # -- 2 & 3. GGUF Upload Logic with Skip Check -------------------------
    quant_configs = [
        {"path": "/kaggle/working/nayari_gguf_q4", "name": "nayari-Q4_K_M.gguf", "method": "q4_k_m"},
        {"path": "/kaggle/working/nayari_gguf_q8", "name": "nayari-Q8_0.gguf",   "method": "q8_0"},
    ]

    for cfg in quant_configs:
        target_name = cfg["name"]
        
        # SKIP LOGIC
        if not REPLACE_EXISTING and target_name in repo_files:
            print(f"Skipping {target_name} (already exists in repo)")
            continue

        local_dir = Path(cfg["path"])
        if not local_dir.exists():
            print(f"\nGenerating {target_name}...")
            # Note: Ensure 'model' and 'tokenizer' are defined in your environment
            if cfg["method"] == "q8_0":
                from unsloth import FastLanguageModel as _FLM
                _m, _t = _FLM.from_pretrained(model_name=MERGED_PATH, max_seq_length=2048, load_in_4bit=False)
                _m.save_pretrained_gguf(cfg["path"], _t, quantization_method=cfg["method"])
                del _m, _t
            else:
                model.save_pretrained_gguf(cfg["path"], tokenizer, quantization_method=cfg["method"])

        files = safe_rename_gguf(cfg["path"], target_name)
        if files:
            f = files[0]
            print(f"Uploading {target_name}...")
            api.upload_file(
                path_or_fileobj=str(f),
                path_in_repo=target_name,
                repo_id=REPO_ID,
                commit_message=f"Add/Update {target_name}",
            )
            print(f"  Done: {target_name}")

print(f"\nAll done! View at https://huggingface.co/{REPO_ID}")



Checking merged 16-bit model folder...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Merged weights synced.

All done! View at https://huggingface.co/Crossie/Nayari


# 2E — HTTP Download Links for ALL GGUF Files
Finds every .gguf file in /kaggle/working, starts a Cloudflare tunnel and prints a direct download URL.
No account needed. Links valid while session is open.

In [7]:
import subprocess, time, socket, re, os, urllib.request
from pathlib import Path
from IPython.display import display, HTML

PORT        = 8989
SERVE_DIR   = "/kaggle/working"
SEARCH_ROOT = Path(SERVE_DIR)
CF_BIN      = "/kaggle/working/cloudflared"

# -- Check internet -------------------------------------------------------
def internet_ok():
    try:
        socket.setdefaulttimeout(5)
        socket.create_connection(("8.8.8.8", 53))
        return True
    except OSError:
        return False

if not internet_ok():
    print("No internet. Enable it in Kaggle Settings, then restart session.")
    raise SystemExit
print("Internet OK")

# -- Find GGUF files ------------------------------------------------------
gguf_files = sorted(SEARCH_ROOT.rglob("*.gguf"))
if not gguf_files:
    print("No .gguf files found. Run Step 9C or 9D first.")
    raise SystemExit

print(f"Found {len(gguf_files)} GGUF file(s):")
for g in gguf_files:
    print(f"   {g.relative_to(SEARCH_ROOT)}  ({g.stat().st_size/1024**3:.2f} GB)")
print()

# -- Kill any old server/tunnel processes ----------------------------------
subprocess.run(["pkill", "-f", f"http.server.*{PORT}"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

# -- Download cloudflared if needed ----------------------------------------
if not Path(CF_BIN).exists():
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN
    ], check=True)
    os.chmod(CF_BIN, 0o755)
    print("cloudflared ready")
else:
    print("cloudflared already present")
    os.chmod(CF_BIN, 0o755)

# -- Start HTTP server and verify it's alive -------------------------------
print(f"\nStarting HTTP server on port {PORT} ...")
server_proc = subprocess.Popen(
    ["python", "-m", "http.server", str(PORT), "--directory", SERVE_DIR,
     "--bind", "0.0.0.0"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait and verify the server is actually responding
server_alive = False
for attempt in range(10):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=3)
        server_alive = True
        break
    except Exception:
        pass

if not server_alive:
    print(f"HTTP server failed to start on port {PORT}!")
    print("Try restarting the Kaggle session.")
    server_proc.terminate()
    raise SystemExit

print(f"HTTP server running and verified (PID {server_proc.pid})")

# -- Start Cloudflare tunnel -----------------------------------------------
print("\nStarting Cloudflare tunnel (may take 15-30s) ...")
cf_log  = "/kaggle/working/cf_tunnel.log"
# Remove old log to avoid matching stale URLs
if Path(cf_log).exists():
    Path(cf_log).unlink()

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
     "--no-autoupdate", "--logfile", cf_log,
     "--protocol", "http2"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(60):  # wait up to 60 seconds
    time.sleep(1)
    if Path(cf_log).exists():
        log_text = Path(cf_log).read_text(errors="ignore")
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log_text)
        if m:
            tunnel_url = m.group(0)
            # Verify the tunnel is actually working
            time.sleep(3)  # give it a moment to stabilize
            try:
                resp = urllib.request.urlopen(tunnel_url, timeout=10)
                if resp.status == 200:
                    break
            except Exception:
                # Tunnel URL found but not yet stable, keep waiting
                pass

if not tunnel_url:
    print("Tunnel URL not found after 60s.")
    if Path(cf_log).exists():
        print("Last 500 chars of tunnel log:")
        print(Path(cf_log).read_text(errors="ignore")[-500:])
    if 'server_proc' in globals(): server_proc.terminate()
    if 'cf_proc' in globals(): cf_proc.terminate()
    raise SystemExit

print(f"Tunnel live: {tunnel_url}\n")

# -- Print download links --------------------------------------------------
rows = ""
print("-- Download links --")
for g in gguf_files:
    rel     = g.relative_to(SEARCH_ROOT)
    size_gb = g.stat().st_size / 1024**3
    url     = f"{tunnel_url}/{rel}"
    print(f"  {g.name}  ({size_gb:.2f} GB)")
    print(f"     {url}")
    print(f'     wget "{url}" -O "{g.name}"')
    print()
    rows += (f"<tr><td><b>{g.name}</b></td><td>{size_gb:.2f} GB</td>"
             f"<td><a href='{url}' target='_blank'>Download</a></td>"
             f'<td><code>wget "{url}" -O "{g.name}"</code></td></tr>')

display(HTML(f"""
<h3>GGUF Download Links (Cloudflare Tunnel)</h3>
<p style='color:orange'>Links are active only while this session is open.</p>
<table border='1' cellpadding='6' cellspacing='0'
       style='border-collapse:collapse;font-family:monospace;font-size:13px'>
  <tr style='background:#f0f0f0'><th>File</th><th>Size</th><th>Link</th><th>wget command</th></tr>
  {rows}
</table>
<p>Browse all: <a href='{tunnel_url}' target='_blank'>{tunnel_url}</a></p>
"""))
print("Keep this notebook session OPEN while downloading.")
print("Run the stop cell below when done.")


Internet OK
Found 2 GGUF file(s):
   nayari_gguf_q4_gguf/Nayari.Q4_K_M.gguf  (0.92 GB)
   nayari_merged_16bit_gguf/nayari_merged_16bit.Q8_0.gguf  (1.53 GB)

cloudflared ready

Starting HTTP server on port 8989 ...
HTTP server running and verified (PID 11016)

Starting Cloudflare tunnel (may take 15-30s) ...
Tunnel live: https://radiation-sub-facility-repository.trycloudflare.com

-- Download links --
  Nayari.Q4_K_M.gguf  (0.92 GB)
     https://radiation-sub-facility-repository.trycloudflare.com/nayari_gguf_q4_gguf/Nayari.Q4_K_M.gguf
     wget "https://radiation-sub-facility-repository.trycloudflare.com/nayari_gguf_q4_gguf/Nayari.Q4_K_M.gguf" -O "Nayari.Q4_K_M.gguf"

  nayari_merged_16bit.Q8_0.gguf  (1.53 GB)
     https://radiation-sub-facility-repository.trycloudflare.com/nayari_merged_16bit_gguf/nayari_merged_16bit.Q8_0.gguf
     wget "https://radiation-sub-facility-repository.trycloudflare.com/nayari_merged_16bit_gguf/nayari_merged_16bit.Q8_0.gguf" -O "nayari_merged_16bit.Q8_0.gguf"

File,Size,Link,wget command
Nayari.Q4_K_M.gguf,0.92 GB,Download,"wget ""https://radiation-sub-facility-repository.trycloudflare.com/nayari_gguf_q4_gguf/Nayari.Q4_K_M.gguf"" -O ""Nayari.Q4_K_M.gguf"""
nayari_merged_16bit.Q8_0.gguf,1.53 GB,Download,"wget ""https://radiation-sub-facility-repository.trycloudflare.com/nayari_merged_16bit_gguf/nayari_merged_16bit.Q8_0.gguf"" -O ""nayari_merged_16bit.Q8_0.gguf"""


Keep this notebook session OPEN while downloading.
Run the stop cell below when done.


# 2F-Weights Download Http - Run to download weights as a zip file

In [17]:
import os
import subprocess
import threading
import re

# --- CONFIG ---
SOURCE_PATH = '/kaggle/working/nayari_merged_16bit'
ZIP_NAME = 'nayari_weights.zip'
PORT = 8000

# 1. Zip the weights (Only if zip doesn't exist)
if not os.path.exists(ZIP_NAME):
    if os.path.exists(SOURCE_PATH):
        print(f"📦 Zipping {SOURCE_PATH}...")
        # -q runs zip in quiet mode for a cleaner log
        !zip -r -q {ZIP_NAME} {SOURCE_PATH}
        print("✅ Zipping complete.")
    else:
        print(f"❌ Error: {SOURCE_PATH} folder not found!")
else:
    print(f"ℹ️ {ZIP_NAME} already exists, skipping zip step.")

# 2. Download Cloudflared
if not os.path.exists('cloudflared'):
    print("📥 Downloading cloudflared...")
    !curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
    !chmod +x cloudflared

# 3. Start Python HTTP Server
def run_server():
    # Serve files from /kaggle/working
    os.chdir('/kaggle/working')
    subprocess.run(["python", "-m", "http.server", str(PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_server, daemon=True).start()

# 4. Start Cloudflare Tunnel and Extract URL using Regex
print("🌉 Opening Cloudflare Tunnel (waiting for URL)...")
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'], 
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = ""
# This regex looks specifically for the trycloudflare URL pattern
url_pattern = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')

for line in iter(proc.stdout.readline, ""):
    # Check for the pattern in the current line
    match = url_pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*60)
        print(f"✨ DOWNLOAD LINK READY ✨")
        print(f"Direct Link: {tunnel_url}/{ZIP_NAME}")
        print("="*60)
        print("\nKeep this cell running until your download finishes.")
        break
    
    # Check if the process died early
    if proc.poll() is not None:
        print("\n❌ Cloudflare tunnel failed to start. Check your internet connection.")
        break

ℹ️ nayari_weights.zip already exists, skipping zip step.
🌉 Opening Cloudflare Tunnel (waiting for URL)...

✨ DOWNLOAD LINK READY ✨
Direct Link: https://insight-dame-element-settings.trycloudflare.com/nayari_weights.zip

Keep this cell running until your download finishes.


# 2G-STOP — Run this to shut down the tunnel + server

In [8]:
if 'server_proc' in globals():
    server_proc.terminate()
if 'cf_proc' in globals():
    cf_proc.terminate()
print("✅ Server and tunnel stopped.")

✅ Server and tunnel stopped.
